# Workflow_n8n

Este sistema es un agente de inteligencia artificial conversacional diseñado para actuar como un ojeador experto y director deportivo del Mundial de Qatar 2022. Para lograrlo, el sistema utiliza una arquitectura basada en LangChain que integra un modelo de lenguaje (Azure OpenAI gpt-4o-mini), memoria de la conversación y un sistema de herramientas que consultan la base de datos abierta de StatsBomb.

El flujo se divide en la arquitectura principal y un sub-flujo de apoyo:

**Flujo Principal**
- Escucha mensajes entrantes desde Telegram y devuelve las respuestas generadas por el agente al mismo chat. 
- Utiliza un Agente que sigue un "Chain of Thought" estricto definido en su System Prompt. 
- Mantiene el contexto de la conversación para poder tener un diálogo continuo con el usuario.

**Tools:**
- Buscador: Una herramienta de peticiones HTTP que el agente usa para buscar el listado general de partidos del Mundial, las alineaciones de un partido concreto...
- Calculadora de estadísticas: Cuando el usuario pide datos estadísticos concretos de un partido, el agente llama a este sub-flujo pasándole el match_id.
    - Procesamiento: Este sub-flujo descarga el archivo completo de eventos de ese partido desde StatsBomb y ejecuta un script para procesar miles de eventos.
    - Salida: Devuelve un resumen limpio al agente indicando cuántos tiros, goles, pases y faltas cometió cada equipo en ese encuentro.

## Configuración de Variables de Entorno

In [ ]:
!az login

In [11]:
import os
import asyncio
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from typing import Annotated
from pydantic import Field
import requests

FOUNDRY_PROJECT_ENDPOINT="https://workflown8n.services.ai.azure.com/"
FOUNDRY_MODEL="gpt-4o-mini"

## Creación de las Tools

In [14]:
@tool(approval_mode="never_require")
def buscar_calendario_o_plantillas(
    endpoint_id: Annotated[str, Field(description="El endpoint exacto a consultar, NUNCA introduzcas solo un número. Formato exacto: 'matches/43/106.json' o 'lineups/NUMERO.json'.")]
) -> dict:
    """
    Herramienta para buscar el calendario o las plantillas/alineaciones.
    """
    url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/{endpoint_id}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        
        # --- FILTRO ANTISATURACIÓN ---
        # Si el modelo pide la lista de partidos, la resumimos drásticamente
        if "matches" in endpoint_id:
            partidos_resumidos = []
            for partido in data:
                partidos_resumidos.append({
                    "match_id": partido.get("match_id"),
                    "fecha": partido.get("match_date"),
                    "fase": partido.get("competition_stage", {}).get("name"),
                    "equipo_local": partido.get("home_team", {}).get("home_team_name"),
                    "equipo_visitante": partido.get("away_team", {}).get("away_team_name"),
                    "goles_local": partido.get("home_score"),
                    "goles_visitante": partido.get("away_score")
                })
            return {"partidos": partidos_resumidos}
            
        # Si pide lineups (plantillas), le devolvemos los datos tal cual
        return data
        
    return {"error": f"Error {response.status_code} al consultar StatsBomb."}

@tool(approval_mode="never_require")
def obtener_estadisticas_partido(
    match_id: Annotated[str, Field(description="El ID numérico del partido. Úsala ÚNICAMENTE para obtener datos estadísticos (tiros, pases, faltas). Ejemplo: '3869685'.")]
) -> dict:
    """
    Procesa los eventos de un partido y devuelve un resumen estadístico básico por equipo.
    """
    url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{match_id}.json"
    response = requests.get(url)
    
    if response.status_code != 200:
        return {"error": "No se encontraron eventos para este partido."}
        
    items = response.json()
    stats = {}
    
    for ev in items:
        if 'team' not in ev:
            continue
            
        team = ev['team']['name']
        
        if team not in stats:
            stats[team] = {"tiros": 0, "goles": 0, "pases": 0, "faltas_cometidas": 0}
            
        ev_type = ev.get('type', {}).get('name')
        if ev_type == "Pass":
            stats[team]["pases"] += 1
        elif ev_type == "Foul Committed":
            stats[team]["faltas_cometidas"] += 1
        elif ev_type == "Shot":
            stats[team]["tiros"] += 1
            if ev.get('shot', {}).get('outcome', {}).get('name') == "Goal":
                stats[team]["goles"] += 1
                
    return {"resumen_estadisticas": stats}

## Creación del Agente

In [26]:
SYSTEM_PROMPT = """
Eres un experto ojeador, analista de datos y director deportivo del Mundial de Fútbol de Qatar 2022. 

### ⚽ GUÍA DE USO DE LA BASE DE DATOS (StatsBomb Open Data vía HTTP Request Tool)
Tienes acceso a los datos oficiales y abiertos de StatsBomb. Para esta base de datos, NO necesitas buscar por fechas ni por nombres de jugador. Todos los datos están estructurados en archivos JSON estáticos.

Utiliza EXACTAMENTE estos formatos para rellenar el parámetro 'endpoint_id' de tu herramienta:

1. Para ver TODOS los partidos y resultados del Mundial 2022:
matches/43/106.json
(Usa siempre este id PRIMERO si te preguntan por el papel de un país, resultados, el calendario o contra quién jugó. Aquí encontrarás el identificador único `match_id` de cada partido).

2. Para ver las Alineaciones y Plantillas completas de un partido concreto:
lineups/{match_id}.json
(Sustituye {match_id} por el ID numérico del partido obtenido en el paso anterior. Este id te dirá la plantilla completa de los dos equipos de ese partido, qué jugadores jugaron, quiénes fueron suplentes, sus posiciones y qué país representan).

3. Para ver las Estadísticas del Partido (Tiros, Pases, Faltas):
Tienes una herramienta secundaria llamada obtener_estadisticas_partido.
(Úsala ÚNICAMENTE cuando el usuario pida datos como quién tiró más, quién dio más pases o pida estadísticas de juego de un partido concreto. Pásale el match_id que hayas conseguido en el paso 1).

### ⚠️ REGLAS ESTRICTAS DE RESPUESTA (CHAIN OF THOUGHT):

1. **REVISIÓN COMPLETA DE PARTIDOS:** Cuando busques los partidos jugados por una selección en `matches/43/106.json`, DEBES leer el archivo completo hasta el final para incluir la Fase de Grupos y las rondas eliminatorias.
2. **INFORME ESTADÍSTICO INTEGRAL:** Cuando uses la herramienta de estadísticas, DEBES presentar un resumen con TODOS los datos (Tiros, Goles, Pases y Faltas) para ambos equipos. 
3. **INTERPRETACIÓN DEL DIRECTOR DEPORTIVO:** OBLIGATORIO añadir un breve párrafo interpretando el partido (quién dominó, efectividad, juego duro...).
4. **FORMATO DE SALIDA:** Escribe en formato de texto normal. ESTÁ PROHIBIDO escribir todo en mayúsculas o usar hashtags (#). Para resaltar, utiliza ÚNICAMENTE negritas con asteriscos (**Texto**).
5. **PROHIBIDO USAR TABLAS:** Telegram no soporta tablas. NUNCA generes tablas de Markdown (con el símbolo |). Si tienes que mostrar datos estadísticos, preséntalos SIEMPRE como una lista de viñetas. 
Usa EXACTAMENTE este formato para las estadísticas:
- **Equipo Local:** X Tiros, X Goles, X Pases, X Faltas.
- **Equipo Visitante:** Y Tiros, Y Goles, Y Pases, Y Faltas.
Sigue estos pasos estrictamente para garantizar la máxima calidad en tu análisis.
"""

client = FoundryChatClient(
    project_endpoint=FOUNDRY_PROJECT_ENDPOINT,
    model=FOUNDRY_MODEL,
    credential=AzureCliCredential(),
)

agent = Agent(
    client=client,
    name="ScoutAgent",
    instructions=SYSTEM_PROMPT,
    tools=[buscar_calendario_o_plantillas, obtener_estadisticas_partido],
)

## Prueba del Agente en la Terminal

In [23]:
async def main() -> None:
    # Creamos la sesión para tener multiconversación (memoria) específica para esta prueba
    session = agent.create_session()

    print("🤖 ScoutAgent inicializado.")
    print("-" * 50)

    # Primera prueba
    pregunta_1 = "¿Contra quién jugó España en el Mundial de Qatar y cuáles fueron los resultados?"
    print(f"👤 Usuario: {pregunta_1}")
    resultado_1 = await agent.run(pregunta_1, session=session)
    print(f"🤖 Agente:\n{resultado_1.text}\n")
    
    print("-" * 50)
    
    # Segunda prueba para comprobar la memoria y el uso de la segunda tool
    pregunta_2 = "De ese primer partido que mencionas de España, ¿quién dio más pases según las estadísticas?"
    print(f"👤 Usuario: {pregunta_2}")
    resultado_2 = await agent.run(pregunta_2, session=session)
    print(f"🤖 Agente:\n{resultado_2.text}\n")

# Ejecutamos la función asíncrona principal
await main()

🤖 ScoutAgent inicializado.
--------------------------------------------------
👤 Usuario: ¿Contra quién jugó España en el Mundial de Qatar y cuáles fueron los resultados?
🤖 Agente:
España jugó un total de tres partidos en la fase de grupos del Mundial de Qatar 2022. Aquí están sus enfrentamientos y resultados:

1. **España vs. Costa Rica**
   - **Fecha:** 23 de noviembre de 2022
   - **Resultado:** España 7 - 0 Costa Rica

2. **España vs. Alemania**
   - **Fecha:** 27 de noviembre de 2022
   - **Resultado:** España 1 - 1 Alemania

3. **Japón vs. España**
   - **Fecha:** 1 de diciembre de 2022
   - **Resultado:** Japón 2 - 1 España

Con estos resultados, España logró avanzar a la siguiente fase del torneo.

--------------------------------------------------
👤 Usuario: De ese primer partido que mencionas de España, ¿quién dio más pases según las estadísticas?
🤖 Agente:
En el partido entre España y Costa Rica, las estadísticas de pases fueron las siguientes:

- **España:**
  - Pases: **109

## Conexión con Telegram

In [27]:
import re
import nest_asyncio
from telegram import Update
from telegram.ext import Application, MessageHandler, filters, ContextTypes

# Aplicamos el parche para que el bot funcione dentro de Jupyter
nest_asyncio.apply()

TELEGRAM_TOKEN = "8223089630:AAEpbJn9iTlmHJkZzsdfw7ylWAikG2ok2KE"

# Diccionario para guardar la memoria (sesión de MAF) de cada usuario independientemente
user_sessions = {}

def formatear(texto: str) -> str:
    """Limpia el texto del LLM y lo prepara para Telegram en formato HTML."""

    # Por si acaso se le escapa algún hashtag, los eliminamos
    texto = re.sub(r'#+\s*', '', texto)
    
    # Convertimos las negritas de Markdown (**texto**) a HTML (<b>texto</b>)
    texto = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', texto)
    
    return texto

async def procesar_mensaje(update: Update, context: ContextTypes.DEFAULT_TYPE):
    chat_id = update.message.chat_id
    user_text = update.message.text
    nombre_usuario = update.message.chat.first_name

    # Si es un usuario nuevo, le creamos una sesión en blanco
    if chat_id not in user_sessions:
        user_sessions[chat_id] = agent.create_session()
        
    session = user_sessions[chat_id]

    # Mostrar el estado "Escribiendo..." en la app de Telegram para dar feedback
    await context.bot.send_chat_action(chat_id=chat_id, action='typing')

    try:
        # Pasamos la pregunta a nuestro agente usando su sesión específica
        resultado = await agent.run(user_text, session=session)
        
        #Formateamos la salida
        texto_limpio = formatear(resultado.text)

        # Devolvemos el mensaje al usuario en Telegram
        await update.message.reply_text(resultado.text)
        print(f"✅ Respuesta enviada a {nombre_usuario}")
        
    except Exception as e:
        print(f"❌ Error al procesar el mensaje: {e}")
        await update.message.reply_text("Uy, he tenido un pequeño fallo técnico analizando los datos. ¿Me lo repites?")

def iniciar_bot():
    if not TELEGRAM_TOKEN:
        print("❌ Error: No se encontró TELEGRAM_BOT_TOKEN.")
        return
        
    # Inicializamos la aplicación de Telegram
    app = Application.builder().token(TELEGRAM_TOKEN).build()
    
    # Le decimos que escuche cualquier texto entrante
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, procesar_mensaje))
    
    print("🤖 ScoutAgent de Telegram en línea y escuchando sin interrupción...")
    print("⚠️ Para apagarlo, pulsa el botón de 'Stop/Interrupt Kernel' en tu notebook.")
    
    # run_polling() inicia el bucle infinito de escucha activa
    app.run_polling()


iniciar_bot()

🤖 ScoutAgent de Telegram en línea y escuchando sin interrupción...
⚠️ Para apagarlo, pulsa el botón de 'Stop/Interrupt Kernel' en tu notebook.
✅ Respuesta enviada a Negerty48
✅ Respuesta enviada a Negerty48
✅ Respuesta enviada a Negerty48
✅ Respuesta enviada a Negerty48


RuntimeError: Cannot close a running event loop